# MODULE 41 — OBJECT DETECTION ARCHITECTURES

## Section 1: Two-Stage Detectors

<badge color="success">Must Know</badge> **Deep Learning — Computer Vision**

Now we're moving from object detection fundamentals to the architectures that actually perform object detection. 🚀

In Module 40, we learned:

- Bounding boxes
- Object localization
- IoU
- Non-Maximum Suppression
- Precision, Recall, AP and mAP

Now let's understand how different neural network architectures use these concepts to detect objects.

---

# 1. What are Two-Stage Detectors?

Imagine you give an image containing cars, pedestrians and bicycles to an object detection model.

How should the model approach the problem?

A two-stage detector divides object detection into two major stages:

1. **Stage 1 — Region Proposal:** Identify regions in the image that might contain objects.
2. **Stage 2 — Object Detection:** Examine those proposed regions, classify the objects and refine their bounding boxes.

![Two-stage object detection concept](https://miro.medium.com/v2/resize%3Afit%3A1018/1%2A24iH9Qi4d9AOzjHmco5g4w.png)

*Reference: [Two-stage detection architecture illustration](https://medium.com/@pramatheshts025/faster-r-cnn-region-proposals-for-high-accuracy-7019001c6e36)*

## Understanding the intuition

Imagine searching for a particular object in a photograph.

Instead of examining every pixel independently:

- First, identify areas that might contain objects.
- Then examine those areas more carefully.
- Finally, determine what each object is and where its boundaries are.

That's the basic idea behind two-stage detection.

### General architecture

```text
             Input Image
                  |
                  ▼
          CNN Feature Extractor
                  |
                  ▼
           Region Proposals
                  |
                  ▼
        Region of Interest (RoI)
                  |
                  ▼
             Detection Head
               /        \
              ▼          ▼
         Classification  Bounding Box
                          Regression
              |               |
              └───────┬───────┘
                      ▼
               Final Detections
```

**Important:** The region proposal stage identifies candidate locations. The detection head determines the object class and improves the bounding box.

---

# 2. R-CNN Family

R-CNN stands for **Region-based Convolutional Neural Network**.

The R-CNN family introduced a major idea: instead of directly predicting objects across an entire image, first identify candidate object regions and then use CNNs to analyze them.

The three important architectures are:

1. R-CNN (2014)
2. Fast R-CNN (2015)
3. Faster R-CNN (2015)

Let's understand how each version improved upon the previous one.

## 2.1 Original R-CNN

![Original R-CNN architecture](https://advaitha.github.io/myDataScienceJourney/Images/R-CNN_transforming_proposal_bb.png)

*Reference: [R-CNN architecture](https://advaitha.github.io/myDataScienceJourney/computer_vision/object_detection/introduction_part2.html)*

The original R-CNN was introduced by Ross Girshick and colleagues in 2014.

Its pipeline:

```text
          Input Image
               |
               ▼
        Selective Search
               |
               ▼
       ~2000 Region Proposals
               |
               ▼
      Crop and Resize Each Region
               |
               ▼
       CNN Feature Extraction
               |
               ▼
       Classification using SVM
               |
               ▼
       Bounding Box Regression
               |
               ▼
         Final Detections
```

Let's break this down.

### Step 1 — Selective Search

Selective Search is a traditional computer vision algorithm that generates candidate object regions.

It uses visual characteristics such as:

- Colour similarity
- Texture similarity
- Region size
- Shape and spatial relationships

It generates approximately 2,000 region proposals per image in the original R-CNN pipeline.

These proposals are candidate regions, not confirmed objects.

![Selective Search region proposals](https://learnopencv.com/wp-content/uploads/2017/09/Selective-Search-Object-Detection.jpg)

*Reference: [LearnOpenCV — Selective Search for Object Detection](https://learnopencv.com/selective-search-for-object-detection-cpp-python/)*

### Step 2 — Crop and Resize

Each proposed region is extracted from the original image and resized to a fixed CNN input size.

**Why?**

CNN architectures generally expect inputs with compatible spatial dimensions.

### Step 3 — CNN Feature Extraction

Each region is passed separately through a CNN.

The CNN extracts visual features such as:

- Edges
- Textures
- Shapes
- Object parts
- Higher-level semantic features

### Step 4 — Classification

The extracted features are passed to class-specific Support Vector Machines (SVMs).

The SVM determines which object class the region belongs to.

### Step 5 — Bounding Box Regression

The predicted bounding box is refined to better fit the actual object's boundaries.

![Bounding box refinement concept](https://miro.medium.com/v2/resize%3Afit%3A1200/1%2AfG6xQm4Wq5cFqWfH5n8h0Q.png)

*Illustration: Bounding-box localization. Reference: [LearnOpenCV](https://learnopencv.com/selective-search-for-object-detection-cpp-python/)*

### Major problem with original R-CNN

The CNN processes every region proposal independently.

If there are approximately 2,000 proposals, the CNN must perform approximately 2,000 forward passes per image.

This creates:

- High computational cost
- Slow inference
- Expensive training
- Large storage requirements for extracted features

**The key limitation:** Repeated CNN feature extraction for overlapping regions.

This leads us to Fast R-CNN.

---

# 3. Fast R-CNN

![Fast R-CNN architecture](https://miro.medium.com/v2/resize%3Afit%3A1400/1%2AJK2zMkomib0o-IjacKoz-A.png)

*Reference: [Fast R-CNN architecture](https://krrishrebba.medium.com/part-6-deep-learning-object-detection-revolution-2014-2025-349425f9616b)*

Fast R-CNN was introduced by Ross Girshick in 2015.

Its biggest improvement:

> Instead of running the CNN separately on every region proposal, run the CNN once on the entire image.

This allows all candidate regions to share the same feature map.

## Fast R-CNN architecture

```text
               Input Image
                    |
                    ▼
               CNN Backbone
                    |
                    ▼
            Shared Feature Map
                    |
                    |
      Selective Search ──┘
            |
            ▼
       Region Proposals
            |
            ▼
         RoI Pooling
            |
            ▼
      Fully Connected Layers
            |
         ┌──┴──────────┐
         ▼             ▼
      Softmax      Bounding Box
    Classification  Regression
         |             |
         └──────┬──────┘
                ▼
         Final Detections
```

Let's understand the improvements.

## 3.1 Shared CNN Feature Map

The entire input image passes through the CNN only once.

The CNN generates a feature map containing spatial and semantic information.

![Fast R-CNN shared feature extraction](https://miro.medium.com/v2/resize%3Afit%3A1400/1%2AJK2zMkomib0o-IjacKoz-A.png)

Now, instead of extracting features separately for every proposed region, we reuse this feature map.

This significantly reduces redundant computation.

## 3.2 Region of Interest (RoI) Pooling

<badge color="success">Very Important Concept</badge>

This is one of the most important concepts in Fast R-CNN.

### What is RoI?

RoI means **Region of Interest**.

It refers to a selected region in the image's feature map corresponding to a candidate object location.

The problem:

Different proposals have different sizes.

For example:

- Region A: 20 × 30
- Region B: 50 × 60
- Region C: 80 × 100

But the fully connected detection head needs a fixed-size feature representation.

RoI Pooling solves this problem.

It converts features from different-sized regions into a fixed spatial size, such as 7 × 7.

![RoI Pooling operation](https://production-media.paperswithcode.com/methods/fast-rcnn.png)

*Reference: [Fast R-CNN — RoI Pooling concept](https://paperswithcode.com/method/fast-r-cnn)*

Conceptually:

```text
      Variable-sized RoI
              |
              ▼
       Divide into fixed grid
              |
              ▼
        Max pooling in each bin
              |
              ▼
       Fixed-size feature map
              |
              ▼
          Detection Head
```

Each bin uses max pooling to produce a fixed-size output.

This makes it possible to feed different-sized candidate regions into the same detection head.

## 3.3 Detection Head

After RoI Pooling, the extracted features are passed through fully connected layers.

The detection head produces two outputs:

### A. Classification head

Predicts the object category.

For example:

```text
Car        → 0.90
Bus        → 0.05
Pedestrian → 0.03
Background → 0.02
```

### B. Bounding box regression head

Predicts adjustments to the proposed bounding box so that it better fits the object.

### What improved over R-CNN?

| Feature | R-CNN | Fast R-CNN |
|---|---|---|
| CNN execution | Separately for each region | Once per image |
| Feature extraction | Repeated | Shared |
| RoI Pooling | No | Yes |
| Classification | SVM | Softmax layer |
| Bounding box regression | Separate stage | Integrated detection head |
| Speed | Slow | Much faster |

But Fast R-CNN still has one major limitation.

It depends on Selective Search to generate region proposals.

Selective Search is an external algorithm, and it can be a computational bottleneck.

Can we replace it with a neural network?

Yes! That's where Faster R-CNN comes in.

---

# 4. Faster R-CNN

<badge color="success">Very Important Architecture</badge>

![Faster R-CNN architecture](https://miro.medium.com/v2/resize%3Afit%3A1018/1%2A24iH9Qi4d9AOzjHmco5g4w.png)

*Reference: [Faster R-CNN architecture](https://medium.com/@pramatheshts025/faster-r-cnn-region-proposals-for-high-accuracy-7019001c6e36)*

Faster R-CNN was introduced by Shaoqing Ren and colleagues in 2015.

Its major improvement:

> It replaces Selective Search with a trainable Region Proposal Network (RPN).

This makes region proposal generation part of the deep learning architecture.

## Faster R-CNN architecture

```text
                 Input Image
                      |
                      ▼
                 CNN Backbone
                      |
                      ▼
               Shared Feature Map
                      |
               ┌──────┴──────┐
               |             |
               ▼             |
              RPN            |
               |             |
               ▼             |
        Region Proposals     |
               |             |
               ▼             |
            RoI Pooling ◄────┘
               |
               ▼
          Detection Head
             /      \
            ▼        ▼
       Classification  Bounding Box
                        Regression
            |               |
            └───────┬───────┘
                    ▼
              Final Detections
```

Let's understand the new component.

## 4.1 Region Proposal Network (RPN)

RPN stands for **Region Proposal Network**.

Its job is to identify regions in the feature map that are likely to contain objects.

Unlike Selective Search, RPN is a neural network trained using object detection data.

![Region Proposal Network architecture](https://production-media.paperswithcode.com/methods/faster_rcnn.png)

*Reference: [Faster R-CNN — Region Proposal Network](https://paperswithcode.com/method/faster-r-cnn)*

### How does RPN work?

#### Step 1 — Input Feature Map

The CNN backbone generates a shared feature map.

#### Step 2 — Anchor Boxes

RPN examines predefined reference boxes called anchors at different spatial locations.

Anchors can have different:

- Scales
- Aspect ratios

For example:

- Small square anchor
- Large square anchor
- Wide rectangular anchor
- Tall rectangular anchor

These anchors provide starting reference boxes for possible objects.

![Anchor boxes at different scales and aspect ratios](https://miro.medium.com/v2/resize%3Afit%3A1200/1%2A5QZgYyH8jWfZzQ5KxH6W5A.png)

*Reference: [Faster R-CNN anchor-box explanation](https://medium.com/@pramatheshts025/faster-r-cnn-region-proposals-for-high-accuracy-7019001c6e36)*

#### Step 3 — Objectness Prediction

RPN predicts whether an anchor is likely to contain an object or background.

For example:

```text
Anchor A → Object: 0.95
Anchor B → Object: 0.80
Anchor C → Object: 0.10
Anchor D → Object: 0.05
```

#### Step 4 — Bounding Box Regression

RPN predicts adjustments to the anchor coordinates.

This helps the proposed bounding box better align with the object.

#### Step 5 — Region Proposals

The adjusted boxes are filtered and ranked. Non-Maximum Suppression removes highly overlapping proposals.

The remaining proposals are passed to the second stage.

## 4.2 Detection Head

The detection head receives the proposed regions and extracts their features.

It predicts:

1. **Object classification** — Is it a car, bus, pedestrian or background?
2. **Bounding box refinement** — How should the proposed coordinates be adjusted?

For example:

```text
       Region Proposal
              |
              ▼
          RoI Features
              |
              ▼
         Detection Head
              |
          ┌───┴──────────┐
          ▼              ▼
      Class: Car    Refined Bounding Box
```

### Important distinction: RPN vs Detection Head

| Component | Responsibility |
|---|---|
| RPN | Generates candidate object regions |
| Detection Head | Classifies proposed regions and refines their boxes |

RPN does not perform the final multi-class object classification. Its objectness prediction answers whether a region is likely to contain an object.

---

# 5. R-CNN vs Fast R-CNN vs Faster R-CNN

Let's compare the evolution.

## R-CNN — 2014

![R-CNN architecture](https://advaitha.github.io/myDataScienceJourney/Images/R-CNN_transforming_proposal_bb.png)

Selective Search generates regions. CNN extracts features separately for every region.

<text color="secondary">Computationally expensive</text>

## Fast R-CNN — 2015

![Fast R-CNN architecture](https://miro.medium.com/v2/resize%3Afit%3A1400/1%2AJK2zMkomib0o-IjacKoz-A.png)

CNN runs once on the image. RoI Pooling extracts fixed-size features for proposals.

<text color="secondary">Shared feature extraction</text>

## Faster R-CNN — 2015

![Faster R-CNN architecture](https://miro.medium.com/v2/resize%3Afit%3A1018/1%2A24iH9Qi4d9AOzjHmco5g4w.png)

RPN replaces external Selective Search with learned region proposals.

<text color="secondary">Learned region proposals</text>

### Complete comparison

| Feature | R-CNN | Fast R-CNN | Faster R-CNN |
|---|---|---|---|
| Region proposal | Selective Search | Selective Search | RPN |
| CNN feature extraction | Per region | Once per image | Once per image |
| RoI Pooling | No | Yes | Yes in original design |
| Detection head | SVM + box regression | Softmax + box regression | Softmax + box regression |
| Proposal generation learned? | No | No | Yes |
| General speed | Slow | Faster | Faster still |

**Note:** Faster R-CNN implementations may use RoI Align instead of the original RoI Pooling operation.

---

# 6. What Exactly Is a Detection Head?

Let's make sure this concept is clear because you'll encounter it in almost every object detection architecture.

A detection head is the part of an object detection model that converts extracted visual features into object predictions.

In two-stage detectors, it operates on features extracted from proposed regions.

It generally performs two main tasks:

## A. Classification

Predicts the object category.

Example:

```text
Car         → 0.85
Bus         → 0.10
Pedestrian  → 0.03
Background  → 0.02
```

## B. Bounding Box Regression

Predicts coordinate adjustments to improve the localization of the object.

Suppose an RoI proposal is:

$$
(100,100,300,300)
$$

The detection head predicts adjustments that produce a refined box closer to the actual object boundaries.

The exact regression parameterization depends on the architecture.

**Remember:** Classification tells us what the object is. Bounding box regression tells us where the object is more accurately.

---

# 7. Must-Know vs Good-to-Know

| Priority | Topics |
|---|---|
| Must Know | R-CNN, Fast R-CNN, Faster R-CNN differences |
| Must Know | Region proposals and RPN |
| Must Know | RoI Pooling |
| Must Know | Classification and bounding box regression heads |
| Good to Know | Selective Search mechanism |
| Good to Know | Anchor boxes, scales and aspect ratios |
| Optional | Detailed original R-CNN training pipeline and loss derivations |

---


# What You Should Remember

- **R-CNN:** Selective Search + separate CNN feature extraction for every region.
- **Fast R-CNN:** Shared CNN feature map + RoI Pooling + detection head.
- **Faster R-CNN:** Replaces Selective Search with a trainable RPN.
- **Region proposals:** Candidate regions that might contain objects.
- **RPN:** Predicts objectness and bounding-box adjustments for anchors.
- **Detection head:** Performs final class prediction and bounding-box refinement.

---